In [2]:
import os
import pandas as pd
import numpy as np

# ========= 路径配置 =========
ALIGN_CSV = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/MS_CXR_combined.csv'  # alignment合并表（含bbox）
REMAINED_CSV = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-remained-data-ap-pa-without-ms-cxr-part.csv'
OUT_CSV = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced.csv'
OUT_STATS = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced_stats.csv'
# ===========================

RSEED = 42
np.random.seed(RSEED)

DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Lung Opacity', 'Pleural Effusion', 'Pneumonia', 'Pneumothorax'
]

# 读数据
align_df = pd.read_csv(ALIGN_CSV)
rem_df   = pd.read_csv(REMAINED_CSV)

# —— 工具：确保有 path（若缺失，用ID构造 files/... 路径）——
def build_rel_path_from_ids(row):
    subj = str(int(row['subject_id'])) if pd.notna(row['subject_id']) else str(row['subject_id'])
    study = str(int(row['study_id'])) if pd.notna(row['study_id']) else str(row['study_id'])
    dicom = str(row['dicom_id'])
    p2 = f"p{str(subj)[:2]}"
    return f"files/{p2}/p{subj}/s{study}/{dicom}.jpg"

def ensure_path_col(df):
    if 'path' not in df.columns:
        if {'subject_id','study_id','dicom_id'}.issubset(df.columns):
            df = df.copy()
            df['path'] = df.apply(build_rel_path_from_ids, axis=1)
        else:
            raise KeyError("缺少 'path' 列，且无法从 subject_id/study_id/dicom_id 构造。")
    return df

align_df = ensure_path_col(align_df)
rem_df   = ensure_path_col(rem_df)

# 只接受 0.0/1.0 的标签（-1、NaN 忽略）
def valid_posneg(series):
    return series.isin([0.0, 1.0])

# 需要的输出列
OUT_COLS = ['dicom_id','subject_id','study_id','target_disease','label','path',
            'x','y','w','h','image_width','image_height']

# 1) 从 alignment 提取“正样本”（每条标注=一行）
pos_parts = []
stats_rows = []

for d in DISEASES:
    # 对齐：只取该病列有效的行，且作为正样本的条件：
    # 1) alignment 的 category_name == d （确保是这个病的标注）
    # 2) 该病列 == 1.0（如果 alignment 表里疾病列存在）
    mask_has_col = (d in align_df.columns)
    if mask_has_col:
        m_valid = valid_posneg(align_df[d])
        m = m_valid & (align_df['category_name'] == d) & (align_df[d] == 1.0)
    else:
        # 如果对齐表没有该病列，则仅用 category_name 过滤
        m = (align_df['category_name'] == d)

    sub = align_df.loc[m, ['dicom_id','subject_id','study_id','path',
                           'x','y','w','h','image_width','image_height']].copy()
    sub['target_disease'] = d
    sub['label'] = 1
    # 排列列顺序
    sub = sub[['dicom_id','subject_id','study_id','target_disease','label','path',
               'x','y','w','h','image_width','image_height']]
    pos_parts.append(sub)

# 合并所有正样本（来自 alignment 的标注）
pos_df = pd.concat(pos_parts, axis=0, ignore_index=True) if pos_parts else \
         pd.DataFrame(columns=OUT_COLS)

# 2) 为每个疾病在 remained 里补齐同数目的“负样本”（0.0）
used_dicoms = set(pos_df['dicom_id'].astype(str))  # 全局去重，先占用所有正样本 dicom
neg_parts = []

for d in DISEASES:
    pos_count = int((pos_df['target_disease'] == d).sum())

    # remained中：该病列==0.0，且有效（过滤-1/NaN）
    if d not in rem_df.columns:
        print(f"[WARN] remained 数据缺失列：{d}，跳过该病负样本补齐。")
        neg_add = 0
    else:
        m_valid = valid_posneg(rem_df[d])
        cand = rem_df.loc[m_valid & (rem_df[d] == 0.0),
                          ['dicom_id','subject_id','study_id','path']].copy()
        # 去掉已使用 dicom（包含所有对病的正与已选负）
        cand = cand[~cand['dicom_id'].astype(str).isin(used_dicoms)]
        # 尽量配平到与正样本同数
        take_n = min(pos_count, len(cand))

        if take_n > 0:
            pick = cand.sample(n=take_n,
                               random_state=abs(hash((d, 'neg', RSEED))) % (2**32-1)).copy()
            pick['target_disease'] = d
            pick['label'] = 0
            # 负样本 bbox 全 NaN
            pick['x'] = np.nan
            pick['y'] = np.nan
            pick['w'] = np.nan
            pick['h'] = np.nan
            pick['image_width'] = np.nan
            pick['image_height'] = np.nan
            pick = pick[['dicom_id','subject_id','study_id','target_disease','label','path',
                         'x','y','w','h','image_width','image_height']]
            neg_parts.append(pick)
            used_dicoms.update(pick['dicom_id'].astype(str).tolist())
            neg_add = take_n
        else:
            neg_add = 0

    stats_rows.append({'disease': d,
                       'pos_from_alignment': pos_count,
                       'neg_added_from_remained': int(neg_add),
                       'final_each_side_target': int(min(pos_count, pos_count))})

# 3) 拼接最终 balanced test（正样本 + 负样本）
neg_df = pd.concat(neg_parts, axis=0, ignore_index=True) if neg_parts else \
         pd.DataFrame(columns=OUT_COLS)
final_df = pd.concat([pos_df, neg_df], axis=0, ignore_index=True)

# 保存
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
final_df.to_csv(OUT_CSV, index=False)

# 统计保存
stats_df = pd.DataFrame(stats_rows)
stats_df['final_pos'] = stats_df['pos_from_alignment']
stats_df['final_neg'] = stats_df['neg_added_from_remained']
stats_df.to_csv(OUT_STATS, index=False)

print(f"[OK] 写入: {OUT_CSV}  行数={len(final_df)}")
print(f"[OK] 统计: {OUT_STATS}")
print(stats_df)


[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced.csv  行数=1988
[OK] 统计: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced_stats.csv
            disease  pos_from_alignment  neg_added_from_remained  \
0       Atelectasis                  49                       49   
1      Cardiomegaly                 151                      151   
2     Consolidation                  78                       78   
3             Edema                  52                       52   
4      Lung Opacity                  70                       70   
5  Pleural Effusion                 135                      135   
6         Pneumonia                 204                      204   
7      Pneumothorax                 255                      255   

   final_each_side_target  final_pos  final_neg  
0                      49         49         49  
1                     151        151        151  
2                      78       

In [4]:
import os
import pandas as pd
import numpy as np

# ===== 路径配置 =====
remained_csv = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/mimic-cxr-remained-data-ap-pa-without-ms-cxr-part.csv'
test_csv     = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced.csv'
out_dir      = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/splits_new'
os.makedirs(out_dir, exist_ok=True)

train_out = os.path.join(out_dir, 'train_part.csv')
val_out   = os.path.join(out_dir, 'val_part.csv')
train_ids_out = os.path.join(out_dir, 'train_part_dicom_ids.txt')
val_ids_out   = os.path.join(out_dir, 'val_part_dicom_ids.txt')

SEED = 42
np.random.seed(SEED)

# 1) 读取
rem_df = pd.read_csv(remained_csv)
test_df = pd.read_csv(test_csv)

# 2) 剔除 test 中涉及的所有 dicom
test_dicoms = set(test_df['dicom_id'].astype(str))
filtered_df = rem_df[~rem_df['dicom_id'].astype(str).isin(test_dicoms)].copy()

print(f"remained 原始行数: {len(rem_df)}")
print(f"test 涉及的 dicom 数: {len(test_dicoms)}")
print(f"剔除后行数: {len(filtered_df)}")

# 3) 以 dicom_id 为单位划分 80/20
all_dicoms = filtered_df['dicom_id'].astype(str).drop_duplicates().tolist()
np.random.shuffle(all_dicoms)  # 打乱
split_idx = int(0.8 * len(all_dicoms))
train_ids = set(all_dicoms[:split_idx])
val_ids   = set(all_dicoms[split_idx:])

# 4) 依据 dicom_id 过滤出行
train_part = filtered_df[filtered_df['dicom_id'].astype(str).isin(train_ids)].copy()
val_part   = filtered_df[filtered_df['dicom_id'].astype(str).isin(val_ids)].copy()

# 5) 一致性检查
assert train_ids.isdisjoint(val_ids), "错误：train/val dicom_id 有交集！"
assert set(train_part['dicom_id']).isdisjoint(set(test_dicoms)), "错误：train 与 test dicom_id 有交集！"
assert set(val_part['dicom_id']).isdisjoint(set(test_dicoms)), "错误：val 与 test dicom_id 有交集！"

print(f"train_part 行数: {len(train_part)} | dicom 数: {train_part['dicom_id'].nunique()}")
print(f"val_part   行数: {len(val_part)}   | dicom 数: {val_part['dicom_id'].nunique()}")

# 6) 保存
train_part.to_csv(train_out, index=False)
val_part.to_csv(val_out, index=False)

# 可选：保存 dicom_id 清单，方便后续复用/核对
with open(train_ids_out, 'w') as f:
    f.write("\n".join(sorted(train_ids)))
with open(val_ids_out, 'w') as f:
    f.write("\n".join(sorted(val_ids)))

print(f"[OK] 写入: {train_out}")
print(f"[OK] 写入: {val_out}")
print(f"[OK] 训练/验证 dicom 清单: {train_ids_out}, {val_ids_out}")


remained 原始行数: 242287
test 涉及的 dicom 数: 1744
剔除后行数: 241293
train_part 行数: 193034 | dicom 数: 193034
val_part   行数: 48259   | dicom 数: 48259
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/splits_new/train_part.csv
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/splits_new/val_part.csv
[OK] 训练/验证 dicom 清单: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/splits_new/train_part_dicom_ids.txt, /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/splits_new/val_part_dicom_ids.txt


In [5]:
import pandas as pd
import numpy as np
import os

# ===== 路径配置 =====
train_part_csv = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/splits_new/train_part.csv'
out_dir = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new'
os.makedirs(out_dir, exist_ok=True)

# 目标规模
TARGET_SIZES = [10_000, 20_000, 50_000, 100_000]

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 疾病列表（无 Lesion）
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Lung Opacity', 'Pleural Effusion', 'Pneumonia', 'Pneumothorax'
]

# 读取数据（新的 train part）
df = pd.read_csv(train_part_csv)

# 仅使用有效标签 0.0 / 1.0（忽略 -1 和 NaN）
def valid_mask(series):
    return series.isin([0.0, 1.0])

# 每个目标规模生成一份数据
for target_total in TARGET_SIZES:
    need_per_side = target_total // (len(DISEASES) * 2)  # 每病每侧目标数
    parts = []
    stats_rows = []

    # 为了不同规模数据的随机性独立，这里给每个规模一个偏移量
    size_seed_offset = target_total

    def rs_for(disease, tag):  # tag: 'pos' / 'neg'
        return abs(hash((disease, tag, RANDOM_SEED, size_seed_offset))) % (2**32 - 1)

    for d in DISEASES:
        if d not in df.columns:
            print(f"[WARN] 训练数据缺列：{d}，该病跳过。")
            stats_rows.append({
                'disease': d,
                'pos_available': 0,
                'neg_available': 0,
                'need_per_side': need_per_side,
                'used_each_side': 0
            })
            continue

        # 有效标签过滤
        valid = valid_mask(df[d])
        pos_df = df.loc[valid & (df[d] == 1.0)].copy()
        neg_df = df.loc[valid & (df[d] == 0.0)].copy()

        n_pos, n_neg = len(pos_df), len(neg_df)
        used_each_side = min(need_per_side, n_pos, n_neg)

        stats_rows.append({
            'disease': d,
            'pos_available': int(n_pos),
            'neg_available': int(n_neg),
            'need_per_side': int(need_per_side),
            'used_each_side': int(used_each_side)
        })

        if used_each_side == 0:
            print(f"[WARN] {d}: 无法构建正负等量（min(pos,neg)=0），该病不纳入本规模。")
            continue

        pos_sample = pos_df.sample(n=used_each_side, replace=False, random_state=rs_for(d, 'pos')).copy()
        neg_sample = neg_df.sample(n=used_each_side, replace=False, random_state=rs_for(d, 'neg')).copy()

        pos_sample['target_disease'] = d
        pos_sample['label'] = 1
        neg_sample['target_disease'] = d
        neg_sample['label'] = 0

        parts.extend([pos_sample, neg_sample])

    # 合并所有疾病
    if parts:
        balanced_df = pd.concat(parts, axis=0, ignore_index=True)
    else:
        balanced_df = pd.DataFrame(columns=['dicom_id','subject_id','study_id','path','target_disease','label'])

    # 只保留常用列（需要更多列可加回）
    keep_cols = ['dicom_id', 'subject_id', 'study_id', 'path', 'target_disease', 'label']
    keep_cols = [c for c in keep_cols if c in balanced_df.columns]
    balanced_df = balanced_df[keep_cols]

    # 打乱
    balanced_df = balanced_df.sample(frac=1.0, random_state=RANDOM_SEED + size_seed_offset).reset_index(drop=True)

    # 实际行数
    actual_total = len(balanced_df)

    # 输出文件名（包含实际大小）
    out_balanced_csv = os.path.join(out_dir, f'balanced_train_{target_total}_actual_{actual_total}.csv')
    out_stats_csv = os.path.join(out_dir, f'balanced_train_{target_total}_stats_actual_{actual_total}.csv')

    # 保存
    balanced_df.to_csv(out_balanced_csv, index=False)
    pd.DataFrame(stats_rows).to_csv(out_stats_csv, index=False)

    print(f'[OK] {target_total}: 保存 {out_balanced_csv}')
    print(f'     实际行数: {actual_total}（理论上最大 {len(DISEASES)*2*need_per_side}）')
    print(f'     统计表:   {out_stats_csv}')


[OK] 10000: 保存 /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train_10000_actual_10000.csv
     实际行数: 10000（理论上最大 10000）
     统计表:   /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train_10000_stats_actual_10000.csv
[OK] 20000: 保存 /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train_20000_actual_19998.csv
     实际行数: 19998（理论上最大 20000）
     统计表:   /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train_20000_stats_actual_19998.csv
[OK] 50000: 保存 /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train_50000_actual_44692.csv
     实际行数: 44692（理论上最大 50000）
     统计表:   /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train_50000_stats_actual_44692.csv
[OK] 100000: 保存 /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new/balanced_train

In [6]:
import os
import glob
import pandas as pd
import numpy as np

# ========= 路径配置 =========
full_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full.parquet'

# 新采样训练 CSV 目录（上一条生成的 balanced_train_*_actual_*.csv）
train_csv_dir = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new'

# 输出 parquet 目录
out_dir = '/home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new_parquet'
os.makedirs(out_dir, exist_ok=True)
# ===========================

# 读取 full.parquet，建立 (dicom_id, disease) -> seed 映射
full_df = pd.read_parquet(full_parquet)

key_cols = ['dicom_id', 'disease', 'seed']
missing = [c for c in key_cols if c not in full_df.columns]
if missing:
    raise RuntimeError(f"full.parquet 缺少列: {missing}")

full_key = (full_df[key_cols]
            .dropna(subset=['dicom_id', 'disease', 'seed'])
            .drop_duplicates())

# groupby 取最小 seed 保持稳定
seed_map = (full_key.groupby(['dicom_id', 'disease'])['seed']
            .apply(lambda s: int(np.min(s.values)))
            .to_dict())

def build_messages_parquet(input_csv: str, split: str, out_parquet: str):
    """
    input_csv: 需包含列 ['dicom_id','target_disease']
    split: 'train'
    out_parquet: 输出 parquet 路径
    """
    df = pd.read_csv(input_csv)
    need_cols = ['dicom_id', 'target_disease']
    miss = [c for c in need_cols if c not in df.columns]
    if miss:
        raise RuntimeError(f"{input_csv} 缺少列: {miss}")

    # 规范类型
    df['dicom_id'] = df['dicom_id'].astype(str)
    df['target_disease'] = df['target_disease'].astype(str)

    # 查 seed（按 (dicom_id, disease)）
    df['seed'] = df.apply(lambda r: seed_map.get((r['dicom_id'], r['target_disease'])), axis=1)

    # 匹配统计与未匹配导出
    matched = df['seed'].notna().sum()
    total = len(df)
    print(f"[{os.path.basename(input_csv)}] matched seeds: {matched}/{total}  (unmatched: {total - matched})")

    if matched < total:
        unmatched_csv = os.path.splitext(out_parquet)[0] + '_unmatched.csv'
        df.loc[df['seed'].isna(), ['dicom_id', 'target_disease']].to_csv(unmatched_csv, index=False)
        print(f"未匹配样本导出: {unmatched_csv}")

    # 仅保留匹配到 seed 的样本
    ok = df[df['seed'].notna()].copy()
    ok['seed'] = ok['seed'].astype(int)

    # 组装目标最简消息格式
    ok['data_source'] = 'cxr_crop'
    ok['prompt'] = ok.apply(lambda _: [{"content": "", "role": "user"}], axis=1)
    ok['extra_info'] = ok.apply(lambda r: {
        "env_config": {"parquet_path": "data/cxr_all/full.parquet"},
        "env_name": "cxr_crop",
        "seed": int(r['seed']),
        "split": split
    }, axis=1)

    out_df = ok[['data_source', 'prompt', 'extra_info']]
    os.makedirs(os.path.dirname(out_parquet), exist_ok=True)
    out_df.to_parquet(out_parquet, index=False)
    print(f"[OK] 写入: {out_parquet}  行数={len(out_df)}")

# 批量处理：匹配四个规模的 CSV
patterns = [
    'balanced_train_10000_actual_*.csv',
    'balanced_train_20000_actual_*.csv',
    'balanced_train_50000_actual_*.csv',
    'balanced_train_100000_actual_*.csv',
]
all_csvs = []
for pat in patterns:
    all_csvs.extend(sorted(glob.glob(os.path.join(train_csv_dir, pat))))

if not all_csvs:
    raise FileNotFoundError(f"未在 {train_csv_dir} 找到 balanced_train_*_actual_*.csv 文件，请检查路径/命名。")

for csv_path in all_csvs:
    base = os.path.splitext(os.path.basename(csv_path))[0]  # 如 balanced_train_10000_actual_9876
    out_parquet = os.path.join(out_dir, base + '.parquet')
    build_messages_parquet(csv_path, split='train', out_parquet=out_parquet)


[balanced_train_10000_actual_10000.csv] matched seeds: 10000/10000  (unmatched: 0)
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new_parquet/balanced_train_10000_actual_10000.parquet  行数=10000
[balanced_train_20000_actual_19998.csv] matched seeds: 19998/19998  (unmatched: 0)
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new_parquet/balanced_train_20000_actual_19998.parquet  行数=19998
[balanced_train_50000_actual_44692.csv] matched seeds: 44692/44692  (unmatched: 0)
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new_parquet/balanced_train_50000_actual_44692.parquet  行数=44692
[balanced_train_100000_actual_82192.csv] matched seeds: 82192/82192  (unmatched: 0)
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/mimic_cxr_jpg_data/train_balanced_new_parquet/balanced_train_100000_actual_82192.parquet  行数=82192


In [7]:
import os
import pandas as pd
import numpy as np

# ===== 输入/输出路径配置 =====
# per-disease 的 test CSV（列：dicom_id,subject_id,study_id,target_disease,label,path）
test_csv = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced.csv'
full_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full.parquet'

# 产出 parquet 路径
out_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_messages.parquet'
# 未匹配导出
out_unmatched_csv = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_messages_unmatched.csv'

# ===== 目标 extra_info 配置 =====
TARGET_PARQUET_PATH = 'data/cxr_all/full.parquet'
SPLIT = 'test'
# =================================

# 1) 读取 full.parquet，构建 (dicom_id, disease) -> seed 映射
full_df = pd.read_parquet(full_parquet)
need_cols = ['dicom_id', 'disease', 'seed']
miss = [c for c in need_cols if c not in full_df.columns]
if miss:
    raise RuntimeError(f"full.parquet 缺少列: {miss}")

full_key = (full_df[need_cols]
            .dropna(subset=need_cols)
            .drop_duplicates())

# 同一个 (dicom_id, disease) 可能多行 —— 取最小 seed 保持稳定
seed_map = (full_key.groupby(['dicom_id', 'disease'])['seed']
            .apply(lambda s: int(np.min(s.values)))
            .to_dict())

# 2) 读取 per-disease test CSV，并按 (dicom_id, target_disease) 查 seed
df = pd.read_csv(test_csv)
req = ['dicom_id', 'target_disease']
miss2 = [c for c in req if c not in df.columns]
if miss2:
    raise RuntimeError(f"{test_csv} 缺少列: {miss2}")

df['dicom_id'] = df['dicom_id'].astype(str)
df['target_disease'] = df['target_disease'].astype(str)

df['seed'] = df.apply(lambda r: seed_map.get((r['dicom_id'], r['target_disease'])), axis=1)

matched = df['seed'].notna().sum()
total = len(df)
print(f"[test] matched seeds: {matched}/{total}  (unmatched: {total - matched})")

# 3) 导出未匹配便于排查
if matched < total:
    df.loc[df['seed'].isna(), ['dicom_id', 'target_disease', 'path']].to_csv(out_unmatched_csv, index=False)
    print(f"未匹配样本导出: {out_unmatched_csv}")

# 4) 仅保留匹配到 seed 的样本并组装 messages
ok = df[df['seed'].notna()].copy()
ok['seed'] = ok['seed'].astype(int)

ok['data_source'] = 'cxr_crop'
ok['prompt'] = ok.apply(lambda _: [{"content": "", "role": "user"}], axis=1)
ok['extra_info'] = ok.apply(lambda r: {
    "env_config": {"parquet_path": TARGET_PARQUET_PATH},
    "env_name": "cxr_crop",
    "seed": int(r['seed']),
    "split": SPLIT
}, axis=1)

out_df = ok[['data_source', 'prompt', 'extra_info']]

os.makedirs(os.path.dirname(out_parquet), exist_ok=True)
out_df.to_parquet(out_parquet, index=False)
print(f"[OK] 写入: {out_parquet}  行数={len(out_df)}")


[test] matched seeds: 1988/1988  (unmatched: 0)
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_messages.parquet  行数=1988


In [8]:
import os
import pandas as pd
import numpy as np

# ===== 路径 =====
test_csv = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_from_alignment_balanced.csv'
full_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/full.parquet'
out_parquet = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_messages_dedup.parquet'
out_unmatched_csv = '/home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_messages_unmatched.csv'

TARGET_PARQUET_PATH = 'data/cxr_all/full.parquet'
SPLIT = 'test'

# 1) (dicom_id, disease) -> seed
full_df = pd.read_parquet(full_parquet)
need_cols = ['dicom_id', 'disease', 'seed']
miss = [c for c in need_cols if c not in full_df.columns]
if miss:
    raise RuntimeError(f"full.parquet 缺少列: {miss}")

full_key = (full_df[need_cols]
            .dropna(subset=need_cols)
            .drop_duplicates())
seed_map = (full_key.groupby(['dicom_id', 'disease'])['seed']
            .apply(lambda s: int(np.min(s.values)))
            .to_dict())

# 2) 读取 test per-disease CSV
df = pd.read_csv(test_csv)
req = ['dicom_id', 'target_disease']
miss2 = [c for c in req if c not in df.columns]
if miss2:
    raise RuntimeError(f"{test_csv} 缺少列: {miss2}")

df['dicom_id'] = df['dicom_id'].astype(str)
df['target_disease'] = df['target_disease'].astype(str)

# 3) 查 seed
df['seed'] = df.apply(lambda r: seed_map.get((r['dicom_id'], r['target_disease'])), axis=1)

# 4) 先去重：每个 (dicom_id, target_disease) 只保留一条（不保留任何 bbox 字段）
df_dedup = df.drop_duplicates(subset=['dicom_id', 'target_disease']).copy()

# 5) 仅保留匹配到 seed 的
matched = df_dedup['seed'].notna().sum()
total = len(df_dedup)
print(f"[test dedup] matched seeds: {matched}/{total}  (unmatched: {total - matched})")

if matched < total:
    df_dedup.loc[df_dedup['seed'].isna(), ['dicom_id', 'target_disease']].to_csv(out_unmatched_csv, index=False)
    print(f"未匹配样本导出: {out_unmatched_csv}")

ok = df_dedup[df_dedup['seed'].notna()].copy()
ok['seed'] = ok['seed'].astype(int)

# 6) 组装最简消息结构（无 bbox）
ok['data_source'] = 'cxr_crop'
ok['prompt'] = ok.apply(lambda _: [{"content": "", "role": "user"}], axis=1)
ok['extra_info'] = ok.apply(lambda r: {
    "env_config": {"parquet_path": TARGET_PARQUET_PATH},
    "env_name": "cxr_crop",
    "seed": int(r['seed']),
    "split": SPLIT
}, axis=1)

out_df = ok[['data_source', 'prompt', 'extra_info']]
os.makedirs(os.path.dirname(out_parquet), exist_ok=True)
out_df.to_parquet(out_parquet, index=False)
print(f"[OK] 写入: {out_parquet}  行数={len(out_df)}")


[test dedup] matched seeds: 1759/1759  (unmatched: 0)
[OK] 写入: /home/aiscuser/verl_new/cxr_data_process/ms_cxr_data/test_messages_dedup.parquet  行数=1759


In [2]:
import pandas as pd


def sample(input_path, output_path, n=20):
    """
    从输入的 parquet 文件中随机采样 20 行，保存为新的 parquet 文件。
    """
    # 读取 parquet
    df = pd.read_parquet(input_path)

    # 随机采样 20 行
    sampled_df = df.sample(n=n, random_state=42)

    # 保存成新的 parquet
    sampled_df.to_parquet(output_path, index=False)

    print(f"保存完成: {output_path}")

# 输入和输出路径
input_path_train = "/home/xufluo/data/cxr_mini_1k_tool/train.parquet"
output_path_train = "/home/xufluo/data/cxr_mini_1k_tool/train_mini.parquet"

sample(input_path_train, output_path_train)

# test
input_path_test = "/home/xufluo/data/cxr_mini_1k_tool/test.parquet"
output_path_test = "/home/xufluo/data/cxr_mini_1k_tool/test_mini.parquet"

sample(input_path_test, output_path_test, n=4)

保存完成: /home/xufluo/data/cxr_mini_1k_tool/train_mini.parquet
保存完成: /home/xufluo/data/cxr_mini_1k_tool/test_mini.parquet
